# Tomografia quântica para N qubits: versão generalizada


In [ ]:
# Código para resolver o problema de tomografia quântica para N q-bits.
import numpy as np
import scipy.optimize as scpo
from scipy.linalg import sqrtm,expm 
import matplotlib.pyplot as plt
import cvxpy as cp
import itertools   
from itertools import product
import time  
from tqdm import tqdm  
import contextlib  
import io
from functools import partial
from tqdm import tqdm

Definimos uma classe para inicializar um estado quântico (trataremos apenas de qubits no trabalho)

In [ ]:
class QuantState:
    def __init__(self, estado):
        """
        Aceita um vetor 1D (estado puro) ou uma matriz 2D (matriz densidade / estado misto).
        """
        estado = np.array(estado, dtype=complex)

        if estado.ndim == 1:
            norma = np.linalg.norm(estado)
            psi = estado / norma
            self.rho = np.outer(psi, psi.conj())

        elif estado.ndim == 2:
            if estado.shape[0] != estado.shape[1]:
                raise ValueError("A matriz densidade deve ser quadrada.")
            traco = np.trace(estado)
            self.rho = estado / traco

        # Preciso definir condições para dimensões maiores
        else:
            raise ValueError("O estado deve ser um vetor 1D (|psi>) ou matriz 2D (rho).")

    def measure(self, obs):
        """
        Mede o estado em relação a um observável (matriz hermitiana).
        Retorna o valor esperado <obs> = Tr(rho * obs).
        """
        return np.trace(self.rho @ obs).real

Como o algoritmo força bruta que fizemos resolve a otimização na matriz T que usamos para garantir que nosso estado seja físico, essas células são para definir matrizes de densidade. Além disso, definimos a função de custo do problema (para ser minimizada), que define os vínculos do problema

In [ ]:
def parametros_para_T_geral(params, n_qubits):
    """Constrói a matriz triangular inferior T para N qubits (D = 2^N)."""
    D = 2**n_qubits
    li, lj = np.tril_indices(D, k=-1)          # índices abaixo da diagonal
    n_off = len(li)                            # D(D-1)/2 elementos complexos

    T = np.zeros((D, D), dtype=complex)
    T[np.arange(D), np.arange(D)] = params[:D]
    T[li, lj] = params[D:D + n_off] + 1j * params[D + n_off:]
    return T


def parametros_para_rho_geral(params, n_qubits):
    """Gera rho para qualquer número de qubits."""
    T = parametros_para_T_geral(params, n_qubits)
    T_dag_T = T.conj().T @ T
    traco = np.trace(T_dag_T).real             #  .real (o traço de T†T é real)

    if np.isclose(traco, 0):
        D = 2**n_qubits
        return np.eye(D, dtype=complex) / D

    return T_dag_T / traco

def funcao_de_custo_geral(params, mediadores_T, dados_exp, n_qubits):
    rho_candidato = parametros_para_rho_geral(params, n_qubits)
    previsoes = np.einsum('ij,mji->m', rho_candidato, mediadores_T).real
    return previsoes - dados_exp

Algumas características do sistema que vamos tratar e são importantes

In [ ]:
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)


# Gera as 4**n - 1 strings de Pauli (todas menos I⊗...⊗I) via produto de Kronecker.
def gerar_paulis(n_qubits):
    I = np.eye(2, dtype=complex)
    base = [I, sigma_x, sigma_y, sigma_z]
    saida = []
    for combo in itertools.product(base, repeat=n_qubits):
        P = combo[0]
        for c in combo[1:]:
            P = np.kron(P, c)
        saida.append(P)
    return saida[1:]        # remove a identidade (o valor esperado dela é sempre 1)


# Fidelidade entre duas matrizes densidade: F = (Tr sqrt( sqrt(A) B sqrt(A) ))^2
def sqrt_psd(A):
    w, V = np.linalg.eigh((A + A.conj().T) / 2)
    return (V * np.sqrt(np.clip(w, 0, None))) @ V.conj().T

def fidelidade(rho_a, rho_b):
    sa = sqrt_psd(rho_a)
    return np.real(np.trace(sqrt_psd(sa @ rho_b @ sa)))**2

Definimos o jacobiano analítico (não entendi de onde vem) para que seja menos custoso nosso programa. Definimos também a função least_squares com os parâmetros que melhor se adequam ao problema (testados em outro código). E definimos também uma função para medir com ruído, simulando uma máquina física real.

In [ ]:
# %% CELULA 1 -- Jacobiano analitico
def jacobiano_analitico(params, mediadores_Q, dados_exp, n_qubits):
    """
    Jacobiano analitico de funcao_de_custo_geral em relacao aos parametros reais.
 
    Deducao (vale reproduzir no relatorio):
      rho(T) = T^dagger T / s,  s = Tr(T^dagger T)
      previsao_m(T) = Tr(rho @ Q_m) = f_m(T)/s,  f_m = Tr(T^dagger T Q_m)
      T e LINEAR em cada parametro real p_i (T = soma_i p_i * B_i, com B_i
      uma matriz-base fixa: E_ii para os p_i diagonais, E_kl para a parte
      real fora da diagonal, i*E_kl para a parte imaginaria). Daqui:
        d f_m/dp_i = 2*Re Tr(T^dagger B_i Q_m)
        d s  /dp_i = 2*Re Tr(T^dagger B_i)
        d previsao_m/dp_i = (2/s) * [Re Tr(T^dagger B_i Q_m) - previsao_m * Re Tr(T^dagger B_i)]
      Usando Tr(T^dagger E_kl Q_m) = (Q_m @ T^dagger)[l,k] e Tr(T^dagger E_kl) = T^dagger[l,k],
      isso fica totalmente vetorizavel (sem laco sobre os parametros).
 
    Validado contra diferenca finita central (erro maximo ~1e-10) para
    n_qubits = 1, 2, 3.
    """
    D = 2 ** n_qubits
    li, lj = np.tril_indices(D, k=-1)
    n_off = len(li)
 
    T = parametros_para_T_geral(params, n_qubits)
    T_dag = T.conj().T
    s = np.trace(T_dag @ T).real
 
    Q = np.asarray(mediadores_Q)
    previsoes = np.einsum('ij,mji->m', T_dag @ T / s, Q).real
 
    M_all = np.einsum('mij,jk->mik', Q, T_dag)   # M_all[m] = Q_m @ T^dagger
 
    tr_diag = np.diagonal(M_all, axis1=1, axis2=2)
    tr_re = M_all[:, lj, li]
    tr_im = 1j * M_all[:, lj, li]
 
    g_diag = np.diagonal(T_dag)
    g_re = T_dag[lj, li]
    g_im = 1j * T_dag[lj, li]
 
    jac_diag = (2.0 / s) * (tr_diag.real - previsoes[:, None] * g_diag.real[None, :])
    jac_re = (2.0 / s) * (tr_re.real - previsoes[:, None] * g_re.real[None, :])
    jac_im = (2.0 / s) * (tr_im.real - previsoes[:, None] * g_im.real[None, :])
 
    return np.hstack([jac_diag, jac_re, jac_im])

def least_squares(fun, x0, **kwargs):
    # Dicionário com os seus parâmetros otimizados
    parametros_padrao = {
        'method': 'dogbox',
        'x_scale': 'jac',
        'tr_solver': 'exact',
        'jac': jacobiano_analitico 
    }
    
    # Atualiza os parâmetros padrão com qualquer outro que você passe na hora
    parametros_padrao.update(kwargs)
    
    # Chama a função original desempaquetando o dicionário
    return scpo.least_squares(fun, x0, verbose=2, **parametros_padrao)


def medir_com_ruido(q_exato, N_shots, rng):
    q_ruido = np.empty_like(q_exato)
    for i, qi in enumerate(q_exato):
        p_mais = (1 + qi) / 2
        outcomes = rng.binomial(N_shots, p_mais)
        q_ruido[i] = 2 * outcomes / N_shots - 1
    return q_ruido

In [ ]:
def roda_com_historico(params0, mediadores_Q, dados_exp, n_qubits, **kwargs_least_squares):
    """
    Roda least_squares com verbose=2 e extrai, da propria saida do scipy, o
    custo reportado a cada ITERACAO aceita (nao a cada avaliacao de funcao --
    isso importa quando o Jacobiano e numerico, porque cada iteracao pode
    embutir varias avaliacoes so para estimar o Jacobiano).
    Retorna (resultado, custos) com custos = array 1D, um valor por iteracao.
    """
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        resultado = least_squares(
            funcao_de_custo_geral,
            x0=params0,
            args=(mediadores_Q, dados_exp, n_qubits),
            **kwargs_least_squares,
        )
    custos = []
    for linha in buf.getvalue().splitlines():
        campos = linha.split()
        if campos and campos[0].lstrip("-").isdigit():
            custos.append(float(campos[2]))
    return resultado, np.array(custos)


SEED_ESTADO = 123
SEED_CHUTE = 456
N_QUBITS_CONTROLE = 3
 
 
def monta_problema_controle(n_qubits=N_QUBITS_CONTROLE):
    rng_estado = np.random.default_rng(SEED_ESTADO)
    rng_chute = np.random.default_rng(SEED_CHUTE)
    D = 2 ** n_qubits
    mediadores_Q = gerar_paulis(n_qubits)
    estado_verdadeiro = QuantState(rng_estado.normal(size=D) + 1j * rng_estado.normal(size=D))
    q_medidos = np.array([estado_verdadeiro.measure(Q) for Q in mediadores_Q])
    chute_inicial = rng_chute.normal(size=D ** 2)
    return dict(n_qubits=n_qubits, mediadores_Q=mediadores_Q, q_medidos=q_medidos,
                chute_inicial=chute_inicial, estado_verdadeiro=estado_verdadeiro)

In [ ]:
def simula_medidas_shots(q_exato, N_shots, rng=None):

    q_exato = np.asarray(q_exato, dtype=float)
    binomial = rng.binomial if rng is not None else np.random.binomial
    q_ruido = np.empty_like(q_exato)
    for i, qi in enumerate(q_exato):
        p_mais = np.clip((1 + qi) / 2, 0.0, 1.0)
        outcomes = binomial(N_shots, p_mais)
        q_ruido[i] = 2 * outcomes / N_shots - 1
    return q_ruido


def medidas_shots_de_estado(estado, mediadores_Q, N_shots, rng=None):

    q_exato = np.array([estado.measure(Q) for Q in mediadores_Q])
    return simula_medidas_shots(q_exato, N_shots, rng=rng)


def resolve_tomografia_lsq(mediadores_Q, q_medidos, n_qubits, chute_inicial=None,
                            estado_verdadeiro=None,
                            **kwargs_least_squares):

    D = 2 ** n_qubits
    if chute_inicial is None:
        chute_inicial = np.random.randn(D ** 2)

    if "tr_solver" not in kwargs_least_squares:
        kwargs_least_squares["tr_solver"] = "lsmr" if D >= 16 else "exact"

    t0 = time.time()

    resultado = least_squares(
        funcao_de_custo_geral, x0=chute_inicial,
        args=(mediadores_Q, q_medidos, n_qubits),
        **kwargs_least_squares
    )

    tempo_s = time.time() - t0

    rho_reconstruido = parametros_para_rho_geral(resultado.x, n_qubits)
    fid = fidelidade(rho_reconstruido, estado_verdadeiro.rho) if estado_verdadeiro is not None else None

    return dict(
        resultado=resultado,
        rho_reconstruido=rho_reconstruido,
        custo_final=resultado.cost,
        nfev=resultado.nfev,
        status=resultado.status,
        mensagem=resultado.message,
        tempo_s=tempo_s,
        fidelidade=fid,
        chute_inicial=chute_inicial,
        q_medidos=np.asarray(q_medidos),
    )


Exeperimento para ver como a resposta do problema se dá com o número de shots que colocamos.

In [ ]:
def experimento_fidelidade_vs_nshots(N_shots_lista=(5, 10, 20, 50, 100, 200, 500, 1000, 5000, 10000, 50000), n_reps=20):
    p = monta_problema_controle()
    rng = np.random.default_rng(999)
    fid_media, fid_std, nfev_media = [], [], []
    for N_shots in N_shots_lista:
        fids, nfevs = [], []
        for _ in range(n_reps):
            q_ruido = medir_com_ruido(p["q_medidos"], N_shots, rng)
            resultado, _ = roda_com_historico(
                p["chute_inicial"], p["mediadores_Q"], q_ruido, p["n_qubits"])
            rho_rec = parametros_para_rho_geral(resultado.x, p["n_qubits"])
            fids.append(fidelidade(rho_rec, p["estado_verdadeiro"].rho))
            nfevs.append(resultado.nfev)
        fid_media.append(np.mean(fids)); fid_std.append(np.std(fids))
        nfev_media.append(np.mean(nfevs))
        print(f"N_shots={N_shots:6d} | fidelidade={np.mean(fids):.4f} +- {np.std(fids):.4f} "
              f"| nfev médio={np.mean(nfevs):.1f}")
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
    ax1.errorbar(N_shots_lista, fid_media, yerr=fid_std, marker="o", capsize=3)
    ax1.set_xscale("log"); ax1.set_xlabel("N_shots"); ax1.set_ylabel("Fidelidade")
    ax1.set_title("Força bruta: fidelidade vs. N_shots"); ax1.grid(alpha=0.3)
    ax2.plot(N_shots_lista, nfev_media, marker="s", color="firebrick")
    ax2.set_xscale("log"); ax2.set_xlabel("N_shots"); ax2.set_ylabel("nfev médio")
    ax2.set_title("Custo do otimizador vs. N_shots (esperado achatado: eixos independentes)")
    ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig("exp7_fidelidade_vs_nshots.png", dpi=140); plt.close()
    return fid_media, fid_std, nfev_media
 

Pronto para rodar para ser exemplo


In [ ]:
if __name__ == "__main__":
    
    print("\n=== Experimento 7: fidelidade vs. N_shots (força bruta) ===")
    experimento_fidelidade_vs_nshots()


Algumas considerações, para o menor numero de iterações:
- Jacobiano Analítico
- mehtod = dogbox
- loss = soft_l1
- x_scale = 'jac'
- tr_solver = 'lsmr'
- diff_step = 1e-8

Pronto para rodar para ser exemplo


In [ ]:
"""Exemplo apenas para teste do algoritmo"""
np.random.seed(41321)  # Para reprodutibilidade
mediadores_Q1 = [sigma_x,  sigma_y, sigma_z]
n1_qubits = 1

unk_state1 = QuantState([2/np.sqrt(5), 1/np.sqrt(5)])

q1_medidos = medidas_shots_de_estado(unk_state1, mediadores_Q1, N_shots=50000)
saida = resolve_tomografia_lsq(mediadores_Q1, q1_medidos, n1_qubits, estado_verdadeiro=unk_state1)
resultado, rho_reconstruido = saida["resultado"], saida["rho_reconstruido"]

print("Rho Reconstruído:\n", rho_reconstruido)
print("Os parâmetros do rho reconstruído são:", np.trace(rho_reconstruido @ sigma_x), np.trace(rho_reconstruido @ sigma_y), np.trace(rho_reconstruido @ sigma_z))
print("Fidelidade entre o estado reconstruído e o estado original:", fidelidade(rho_reconstruido, unk_state1.rho)) 

Pronto para rodar para ser exemplo


In [ ]:
from bloch_reconstrucao import plotar_bloch
np.random.seed(413453432) 
n2_qubits = 1

num_params = (2**n2_qubits)**2 
chute_inicial1 = np.random.randn(num_params)
chute_inicial2 = np.random.randn(num_params)

mediadores_Q2 = [sigma_x, sigma_y]
q_medidos1 = simula_medidas_shots([0.7, 0.4], N_shots=50000)
saida1 = resolve_tomografia_lsq(mediadores_Q2, q_medidos1, n2_qubits, chute_inicial=chute_inicial1, estado_verdadeiro=None)
saida2 = resolve_tomografia_lsq(mediadores_Q2, q_medidos1, n2_qubits, chute_inicial=chute_inicial2, estado_verdadeiro=None)
resultado1, rho_reconstruido1 = saida1["resultado"], saida1["rho_reconstruido"]
resultado2, rho_reconstruido2 = saida2["resultado"], saida2["rho_reconstruido"]

plotar_bloch(
    [rho_reconstruido1, rho_reconstruido2],
    rotulos=["Reconstrução 1", "Reconstrução 2"],
    medidos=("x", "y"),          # destaca o plano dos observáveis que você mediu
    salvar="bloch_reconstrucoes_shots.png",
)


Pronto para rodar para ser exemplo


In [ ]:
np.random.seed(74564089)  # Para reprodutibilidade
n3_qubits = 4         
D = 2**n3_qubits

todos_Q = gerar_paulis(n3_qubits)

n_medidas = len(todos_Q)
mediadores_Q3 = todos_Q[:n_medidas]          

unk_state = QuantState(np.random.randn(D) + 1j * np.random.randn(D))

q_medidos3 = medidas_shots_de_estado(unk_state, mediadores_Q3, N_shots=50000) 

t0 = time.time() 
saida = resolve_tomografia_lsq(mediadores_Q3, q_medidos3, n3_qubits, estado_verdadeiro=unk_state, ftol=1e-6, xtol=1e-6, gtol=1e-8)
print(f"Tempo de uma otimização: {time.time() - t0:.2f} s")  

resultado1, rho_reconstruido1 = saida["resultado"], saida["rho_reconstruido"]
print(f"Tempo: {saida['tempo_s']:.2f}s | Custo: {saida['custo_final']:.3e} | Fidelidade: {saida['fidelidade']:.6f}")

In [ ]:
def roda_benchmark_qubits(n_qubits_list, N_shots=50000, seed_estado=123, seed_chute=456,
                           **kwargs_ls):
    """
    Roda a tomografia para cada valor de n_qubits em n_qubits_list e coleta:
      - fidelidade em relacao ao estado verdadeiro
      - tempo de execucao (s)
      - numero de iteracoes aceitas (via roda_com_historico)
      - nfev, custo final e status do least_squares
 
    seed_estado/seed_chute sao deslocados por n_qubits para que cada tamanho
    de problema tenha uma instancia diferente (mas reprodutivel) de estado e
    chute inicial.
 
    kwargs_ls sao repassados para roda_com_historico / least_squares
    (ex.: ftol, xtol, gtol, max_nfev). Se 'tr_solver' nao for passado, usa
    'lsmr' para D >= 16 e 'exact' caso contrario (mesma regra de
    resolve_tomografia_lsq).
 
    Retorna uma lista de dicts, um por n_qubits.
    """
    registros = []
 
    for n_qubits in tqdm(n_qubits_list, desc="Rodando benchmarks"):
        D = 2 ** n_qubits
        rng_estado = np.random.default_rng(seed_estado + n_qubits)
        rng_chute = np.random.default_rng(seed_chute + n_qubits)
 
        mediadores_Q = gerar_paulis(n_qubits)
        estado = QuantState(rng_estado.normal(size=D) + 1j * rng_estado.normal(size=D))
        q_medidos = medidas_shots_de_estado(estado, mediadores_Q, N_shots=N_shots, rng=rng_estado)
        chute_inicial = rng_chute.normal(size=D ** 2)
 
        kwargs_ls_run = dict(kwargs_ls)
        kwargs_ls_run.setdefault('tr_solver', 'lsmr' if D >= 16 else 'exact')
 
        t0 = time.time()
        resultado, custos = roda_com_historico(
            chute_inicial, mediadores_Q, q_medidos, n_qubits, **kwargs_ls_run
        )
        tempo_s = time.time() - t0
 
        rho_reconstruido = parametros_para_rho_geral(resultado.x, n_qubits)
        fid = fidelidade(rho_reconstruido, estado.rho)
        n_iteracoes = len(custos)
 
        registro = dict(
            n_qubits=n_qubits,
            D=D,
            fidelidade=fid,
            tempo_s=tempo_s,
            n_iteracoes=n_iteracoes,
            nfev=resultado.nfev,
            custo_final=resultado.cost,
            status=resultado.status,
            mensagem=resultado.message,
        )
        registros.append(registro)
 
        print(f"n_qubits={n_qubits:2d} | D={D:4d} | "
              f"Fidelidade={fid:.6f} | Tempo={tempo_s:8.2f}s | "
              f"Iteracoes={n_iteracoes:5d} | nfev={resultado.nfev:5d} | "
              f"status={resultado.status} ({resultado.message})")
        
        tqdm.write(f"n_qubits={n_qubits:2d} | D={D:4d} | "
                   f"Fidelidade={fid:.6f} | Tempo={tempo_s:8.2f}s | "
                   f"Iteracoes={n_iteracoes:5d} | nfev={resultado.nfev:5d} | "
                   f"status={resultado.status} ({resultado.message})")
 
    return registros
 
 
# %% CELULA -- Plots individuais
 
def plota_fidelidade_x_qubits(registros, ax=None):
    qubits = [r["n_qubits"] for r in registros]
    fids = [r["fidelidade"] for r in registros]
 
    ax_local = ax if ax is not None else plt.figure(figsize=(6, 4)).gca()
    ax_local.plot(qubits, fids, marker="o")
    ax_local.set_xlabel("Numero de qubits")
    ax_local.set_ylabel("Fidelidade")
    ax_local.set_title("Fidelidade vs. numero de qubits")
    ax_local.set_xticks(qubits)
    ax_local.grid(True, alpha=0.3)
    if ax is None:
        plt.tight_layout()
        plt.show()
 
 
def plota_tempo_x_qubits(registros, ax=None, log_y=False):
    qubits = [r["n_qubits"] for r in registros]
    tempos = [r["tempo_s"] for r in registros]
 
    ax_local = ax if ax is not None else plt.figure(figsize=(6, 4)).gca()
    ax_local.plot(qubits, tempos, marker="o", color="tab:orange")
    if log_y:
        ax_local.set_yscale("log")
    ax_local.set_xlabel("Numero de qubits")
    ax_local.set_ylabel("Tempo (s)" + (" [escala log]" if log_y else ""))
    ax_local.set_title("Tempo de execucao vs. numero de qubits")
    ax_local.set_xticks(qubits)
    ax_local.grid(True, alpha=0.3, which="both")
    if ax is None:
        plt.tight_layout()
        plt.show()
 
 
def plota_iteracoes_x_qubits(registros, ax=None, log_y=False):
    qubits = [r["n_qubits"] for r in registros]
    iters = [r["n_iteracoes"] for r in registros]
 
    ax_local = ax if ax is not None else plt.figure(figsize=(6, 4)).gca()
    ax_local.plot(qubits, iters, marker="o", color="tab:green")
    if log_y:
        ax_local.set_yscale("log")
    ax_local.set_xlabel("Numero de qubits")
    ax_local.set_ylabel("Numero de iteracoes" + (" [escala log]" if log_y else ""))
    ax_local.set_title("Iteracoes vs. numero de qubits")
    ax_local.set_xticks(qubits)
    ax_local.grid(True, alpha=0.3, which="both")
    if ax is None:
        plt.tight_layout()
        plt.show()
 
 
def plota_tudo(registros):
    """Os tres graficos lado a lado numa figura so."""
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    plota_fidelidade_x_qubits(registros, ax=axs[0])
    plota_tempo_x_qubits(registros, ax=axs[1])
    plota_iteracoes_x_qubits(registros, ax=axs[2])
    plt.tight_layout()
    plt.show()

Pronto para rodar para ser exemplo

In [ ]:
"Teste Qubits x Fidelidade/Tempo/Iteracoes"
n_qubits_list = [1, 2, 3, 4, 5, 6]

registros = roda_benchmark_qubits(
    n_qubits_list,
    N_shots=50000,
    ftol=1e-6, xtol=1e-6, gtol=1e-8,
)

plota_fidelidade_x_qubits(registros)
plota_tempo_x_qubits(registros)

In [ ]:
def gera_estado_teste(tipo, n_qubits, rng, beta_gibbs=1.0, hamiltoniano=None):
    """
    Gera um QuantState de um dos seguintes tipos, para investigar como o
    comportamento numerico do SDP muda com a estrutura fisica do estado:

      'puro'                 -> estado puro aleatorio (posto 1)
      'mistura_dois_puros'   -> combinacao convexa de 2 estados puros (posto <= 2)
      'muito_misto'          -> mistura de muitos estados aleatorios com pesos
                                 quase uniformes (perto do maximamente misto,
                                 posto cheio, bem no interior do cone PSD)
      'gibbs'                -> estado termico exp(-beta H)/Z de um Hamiltoniano
                                 hermitiano aleatorio (ou fornecido). beta grande
                                 (baixa T) aproxima do estado fundamental (quase
                                 puro); beta pequeno (alta T) aproxima do
                                 maximamente misto.
    """
    D = 2 ** n_qubits

    if tipo == "puro":
        psi = rng.normal(size=D) + 1j * rng.normal(size=D)
        return QuantState(psi)

    elif tipo == "mistura_dois_puros":
        psi1 = rng.normal(size=D) + 1j * rng.normal(size=D)
        psi2 = rng.normal(size=D) + 1j * rng.normal(size=D)
        rho1 = QuantState(psi1).rho
        rho2 = QuantState(psi2).rho
        p = rng.uniform(0.3, 0.7)
        return QuantState(p * rho1 + (1 - p) * rho2)

    elif tipo == "muito_misto":
        n_componentes = max(4 * D, 16)
        # concentracao alta no Dirichlet -> pesos quase uniformes -> perto
        # do maximamente misto sem ser exatamente ele (que seria trivial)
        pesos = rng.dirichlet(np.ones(n_componentes) * 50)
        rho = np.zeros((D, D), dtype=complex)
        for p in pesos:
            psi = rng.normal(size=D) + 1j * rng.normal(size=D)
            rho += p * QuantState(psi).rho
        return QuantState(rho)

    elif tipo == "gibbs":
        if hamiltoniano is None:
            H = rng.normal(size=(D, D)) + 1j * rng.normal(size=(D, D))
            H = (H + H.conj().T) / 2  # forca hermitiana
        else:
            H = hamiltoniano
        rho = expm(-beta_gibbs * H)
        return QuantState(rho)  # QuantState ja normaliza pelo traco

    else:
        raise ValueError(f"tipo de estado desconhecido: {tipo!r}")


In [ ]:
def resolve_tomografia_sdp(mediadores_Q, q_medidos, n_qubits, solver=None, **solver_kwargs):
    """
    Versao SDP de minimos quadrados: minimiza sum_i (Tr(Q_i rho) - q_i)^2
    sujeito a rho >> 0, Tr(rho) = 1. Convexo -> minimo global garantido,
    sem risco de minimo local (diferente do least_squares nao-linear).

    solver: string do cvxpy (ex. 'SCS', 'CLARABEL', 'MOSEK'). None deixa o
    cvxpy escolher automaticamente.
    solver_kwargs: repassado para problema.solve(...) (ex. verbose=True,
    max_iters=..., eps=...).
    """
    D = 2 ** n_qubits
    Q = mediadores_Q
    q = np.asarray(q_medidos, dtype=float)
    M = len(Q)

    rho_var = cp.Variable((D, D), hermitian=True)
    previsoes = cp.hstack([cp.real(cp.trace(Q[i] @ rho_var)) for i in range(M)])
    objetivo = cp.Minimize(cp.sum_squares(previsoes - q))
    restricoes = [rho_var >> 0, cp.trace(rho_var) == 1]
    problema = cp.Problem(objetivo, restricoes)

    t0 = time.time()
    problema.solve(solver=solver, **solver_kwargs)
    tempo_s = time.time() - t0

    stats = problema.solver_stats
    n_iter = getattr(stats, "num_iters", None) if stats is not None else None
    solver_name = getattr(stats, "solver_name", None) if stats is not None else None

    return dict(
        problema=problema,
        rho_reconstruido=rho_var.value,
        status=problema.status,
        valor_otimo=problema.value,
        tempo_s=tempo_s,
        n_iteracoes=n_iter,
        solver_name=solver_name,
    )

In [ ]:
def roda_benchmark_qubits_sdp(n_qubits_list, tipo_estado="puro", N_shots=50000,
                               seed_estado=123, solver=None, beta_gibbs=1.0,
                               **solver_kwargs):
    """
    Mesma ideia do benchmark do least_squares nao-linear (roda_benchmark_qubits),
    so que resolvendo via SDP. Os registros tem os MESMOS nomes de campo
    (n_qubits, fidelidade, tempo_s, n_iteracoes), entao as funcoes de plot
    plota_fidelidade_x_qubits / plota_tempo_x_qubits / plota_iteracoes_x_qubits
    (do benchmark_qubits.py) funcionam direto em cima do resultado daqui.

    CUIDADO: SDP escala mal com D = 2^n_qubits (a variavel e uma matriz D x D
    com restricao de semidefinita positividade). Com solver generico (SCS),
    D=16 (4 qubits) ja costuma ficar bem mais lento que o least_squares
    nao-linear equivalente; D=32 (5 qubits) pode ficar pesado. Comece
    pequeno (ate 3-4 qubits) antes de escalar.
    """
    registros = []

    for n_qubits in n_qubits_list:
        rng_estado = np.random.default_rng(seed_estado + n_qubits)
        mediadores_Q = gerar_paulis(n_qubits)
        estado = gera_estado_teste(tipo_estado, n_qubits, rng_estado, beta_gibbs=beta_gibbs)
        q_medidos = medidas_shots_de_estado(estado, mediadores_Q, N_shots=N_shots, rng=rng_estado)

        saida = resolve_tomografia_sdp(mediadores_Q, q_medidos, n_qubits,
                                        solver=solver, **solver_kwargs)
        fid = fidelidade(estado.rho, saida["rho_reconstruido"]) if saida["rho_reconstruido"] is not None else np.nan

        # posto numerico do estado verdadeiro, so para contextualizar o print
        autovalores = np.linalg.eigvalsh(estado.rho)
        posto_efetivo = int(np.sum(autovalores > 1e-6))

        registro = dict(
            n_qubits=n_qubits, D=2 ** n_qubits, tipo_estado=tipo_estado,
            fidelidade=fid, tempo_s=saida["tempo_s"], n_iteracoes=saida["n_iteracoes"],
            status=saida["status"], valor_otimo=saida["valor_otimo"],
            solver=saida["solver_name"], posto_efetivo=posto_efetivo,
        )
        registros.append(registro)

        print(f"[{tipo_estado:20s}] n_qubits={n_qubits} | D={2**n_qubits:3d} | "
              f"posto~{posto_efetivo:2d} | Fidelidade={fid:.6f} | "
              f"Tempo={saida['tempo_s']:7.3f}s | Iteracoes={saida['n_iteracoes']} | "
              f"status={saida['status']} | solver={saida['solver_name']}")

    return registros


In [ ]:



def roda_sdp_com_historico_scs(mediadores_Q, q_medidos, n_qubits, **solver_kwargs):
    D = 2 ** n_qubits
    Q = mediadores_Q
    q = np.asarray(q_medidos, dtype=float)
    M = len(Q)

    rho_var = cp.Variable((D, D), hermitian=True)
    previsoes = cp.hstack([cp.real(cp.trace(Q[i] @ rho_var)) for i in range(M)])
    problema = cp.Problem(
        cp.Minimize(cp.sum_squares(previsoes - q)),
        [rho_var >> 0, cp.trace(rho_var) == 1],
    )

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        problema.solve(solver="SCS", verbose=True, **solver_kwargs)

    objetivos = []
    for linha in buf.getvalue().splitlines():
        campos = linha.split("|")
        # tabela do SCS: "  it | pri res | dua res | ... | objective | ..."
        # ajuste os indices se a versao do SCS instalada formatar diferente
        if len(campos) >= 6 and campos[0].strip().isdigit():
            try:
                objetivos.append(float(campos[-2].strip()))
            except ValueError:
                pass

    return problema, rho_var, np.array(objetivos)

In [ ]:

n_qubits_list = [1, 2, 3, 4]

reg_puro   = roda_benchmark_qubits_sdp(n_qubits_list, tipo_estado="puro")
reg_mist2  = roda_benchmark_qubits_sdp(n_qubits_list, tipo_estado="mistura_dois_puros")
reg_misto  = roda_benchmark_qubits_sdp(n_qubits_list, tipo_estado="muito_misto")
reg_gibbs_quente = roda_benchmark_qubits_sdp(n_qubits_list, tipo_estado="gibbs", beta_gibbs=0.1)
reg_gibbs_frio   = roda_benchmark_qubits_sdp(n_qubits_list, tipo_estado="gibbs", beta_gibbs=10.0)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for reg, label in [(reg_puro, "puro"), (reg_mist2, "mistura 2 puros"),
                    (reg_misto, "muito misto"),
                    (reg_gibbs_quente, "gibbs (alta T)"),
                    (reg_gibbs_frio, "gibbs (baixa T)")]:
    qubits = [r["n_qubits"] for r in reg]
    axs[1].plot(qubits, [r["tempo_s"] for r in reg], marker="o", label=label)
    axs[2].plot(qubits, [r["n_iteracoes"] for r in reg], marker="o", label=label)
axs[0].set_title("Fidelidade"); axs[1].set_title("Tempo (s)"); axs[2].set_title("Iteracoes")
for ax in axs:
    ax.set_xlabel("n_qubits"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
"""Tentando fazer usando SDP"""
# Exemplo Sanity Check (assim como o primeiro, apenas para testar e ver se funciona)
n_qubits = 1
d = 2**n_qubits                 # dimensão da matriz densidade
M = len(mediadores_Q1)       # número de mediadores
rho = cp.Variable((d, d), hermitian=True)

Q = mediadores_Q1            # observáveis
q = q1_medidos               # valores esperados das medições

# Vínculos do problema
constraints = [
    rho >> 0,
    cp.trace(rho) == 1
]
for i in range(M):
    constraints.append(
        cp.real(cp.trace(Q[i] @ rho)) == q[i]
    )

# Problema de viabilidade semidefinida
problem = cp.Problem(cp.Minimize(0), constraints)
problem.solve()
print(problem.status)
# Caso haja estado físico real compatível com as medidas
if problem.status == "optimal" and n_qubits <= 2:
    print(np.round(rho.value, 3))
print("SDP viability:", problem.status, "| valor ótimo:", problem.value)
print("Fidelidade com o estado verdadeiro:", round(fidelidade(unk_state1.rho, rho.value), 6))
"""
Pode haver ruído estatístico nos dados, que faz com que 
o problema de viabilidade seja infactível. Nesse caso, 
podemos usar uma versão de mínimos quadrados (convexa) 
para encontrar a melhor aproximação.
"""

# Versão SDP de mínimos quadrados (convexa): minimiza sum (Tr(Q rho) - q)^2
# com rho >= 0 e Tr(rho) = 1. Tem ótimo global (sem mínimo local) e aguenta ruído.
rho_ls = cp.Variable((d, d), hermitian=True)
previsoes = cp.hstack([cp.real(cp.trace(Q[i] @ rho_ls)) for i in range(M)])
problem_ls = cp.Problem(
    cp.Minimize(cp.sum_squares(previsoes - q)),
    [rho_ls >> 0, cp.trace(rho_ls) == 1],
)
problem_ls.solve()
print("SDP mínimos quadrados:", problem_ls.status, "| valor ótimo:", problem_ls.value)
print("Fidelidade com o estado verdadeiro:", round(fidelidade(unk_state1.rho, rho_ls.value), 6))

In [ ]:
"""Mais uma abordagem para a não unicidade da reconstrução de estados quânticos a partir de medidas incompletas, agora usando SDP.0"""
n5_qubits = 1
d = 2**n5_qubits                 # dimensão da matriz densidade
Q1 = mediadores_Q2
q1 = [0.7, 0.4]

M1 = len(Q1)

rho1 = cp.Variable((d, d), hermitian=True)
rho2 = cp.Variable((d, d), hermitian=True)

constraints1 = [
    rho1 >> 0,
    cp.trace(rho1) == 1
]

constraints2 = [   
    rho2 >> 0,
    cp.trace(rho2) == 1
]

for i in range(M1):
    constraints1.append(
        cp.real(cp.trace(Q1[i] @ rho1)) == q1[i]
    )
    constraints2.append(
        cp.real(cp.trace(Q1[i] @ rho2)) == q1[i]
    )

problem1 = cp.Problem(cp.Minimize(0), constraints1)
problem2 = cp.Problem(cp.Minimize(0), constraints2)  # Mesma configuração para o segundo problema
problem1.solve()
problem2.solve()

print("SDP Viability 1:", problem1.status, "; SDP Viability 2:", problem2.status)
print("Rho Reconstruído 1:\n", np.round(rho1.value, 3))
print("Rho Reconstruído 2:\n", np.round(rho2.value, 3))

# Entender o porquê dá igual 
# Para mostrar a não unicidade, temos que determinar algum parâmetro 
# para diferenciar os dois estados reconstruídos. Vamos maximizar o 
# sigma_z para o primeiro estado e minimizar para o segundo, mantendo 
# as mesmas medidas, assim, pelo menos graficamente, podemos ver que são 
# diferentes, e com uma imagem melhor.



In [ ]:
"""
Utilizar ruído para mostrar que o problema de viabilidade pode ser infactível, 
e que a versão de mínimos quadrados (convexa) ainda encontra uma boa aproximação.
"""

"""
Exemplo: efeito do ruido de shot-noise na tomografia via SDP.

Ideia: em vez de assumir que as medicoes q_i = Tr(Q_i rho) sao exatas,
simulamos que cada uma vem de N_shots medicoes binomiais (regra de Born).
Para N_shots pequeno, o problema de viabilidade (Minimize(0)) costuma ficar
infactivel; a versao de minimos quadrados sempre devolve uma resposta, e sua
fidelidade com o estado verdadeiro cresce conforme N_shots aumenta.

Ajuste os nomes de variaveis (Q, q, fidelidade, unk_state, mediadores_Q2 etc.)
para bater com o resto do seu notebook, se ja tiver essas coisas definidas la.
"""


rng = np.random.default_rng(42)

# --- 1. Estado "verdadeiro" que queremos recuperar (1 qubit) --------------
# Troque por unk_state.rho se ja tiver um estado definido no seu notebook.
d = 2
G = rng.normal(size=(d, d)) + 1j * rng.normal(size=(d, d))
rho_true = G @ G.conj().T
rho_true /= np.trace(rho_true).real

sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
Q = [sx, sy, sz]           # troque por mediadores_Q2 se quiser reusar o seu
M = len(Q)

q_exato = np.array([np.trace(Qi @ rho_true).real for Qi in Q])

def medir_com_ruido(q_exato, N_shots, rng):
    """Simula N_shots medicoes binomiais por observavel (autovalores +-1)."""
    q_ruido = np.empty_like(q_exato)
    for i, qi in enumerate(q_exato):
        p_mais = (1 + qi) / 2
        outcomes = rng.binomial(N_shots, p_mais)
        q_ruido[i] = 2 * outcomes / N_shots - 1
    return q_ruido


def resolver_viabilidade(Q, q_ruido, d):
    rho_var = cp.Variable((d, d), hermitian=True)
    constraints = [rho_var >> 0, cp.trace(rho_var) == 1]
    for i in range(len(Q)):
        constraints.append(cp.real(cp.trace(Q[i] @ rho_var)) == q_ruido[i])
    problem = cp.Problem(cp.Minimize(0), constraints)
    problem.solve(solver=cp.SCS)
    return problem, rho_var


def resolver_minimos_quadrados(Q, q_ruido, d):
    rho_var = cp.Variable((d, d), hermitian=True)
    previsoes = cp.hstack([cp.real(cp.trace(Q[i] @ rho_var)) for i in range(len(Q))])
    problem = cp.Problem(
        cp.Minimize(cp.sum_squares(previsoes - q_ruido)),
        [rho_var >> 0, cp.trace(rho_var) == 1],
    )
    problem.solve(solver=cp.SCS)
    return problem, rho_var


N_shots_lista = [10, 20, 50, 100, 200, 500, 1000, 5000, 10000]
n_reps = 30

fidelidades_media = []
fidelidades_std = []
taxa_infactivel = []

for N_shots in N_shots_lista:
    fids = []
    n_infactivel = 0
    for _ in range(n_reps):
        q_ruido = medir_com_ruido(q_exato, N_shots, rng)

        prob_viab, _ = resolver_viabilidade(Q, q_ruido, d)
        if prob_viab.status != "optimal":
            n_infactivel += 1

        prob_ls, rho_ls = resolver_minimos_quadrados(Q, q_ruido, d)
        fids.append(fidelidade(rho_true, rho_ls.value))

    fidelidades_media.append(np.mean(fids))
    fidelidades_std.append(np.std(fids))
    taxa_infactivel.append(n_infactivel / n_reps)

    print(
        f"N_shots={N_shots:6d} | fidelidade media={np.mean(fids):.4f} "
        f"+- {np.std(fids):.4f} | infactivel em {n_infactivel}/{n_reps} repeticoes"
    )

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ax1.errorbar(N_shots_lista, fidelidades_media, yerr=fidelidades_std,
             marker="o", capsize=3)
ax1.set_xscale("log")
ax1.set_xlabel("Numero de shots (N)")
ax1.set_ylabel("Fidelidade com o estado verdadeiro")
ax1.set_title("Minimos quadrados: fidelidade vs. N_shots")
ax1.grid(alpha=0.3)

ax2.plot(N_shots_lista, taxa_infactivel, marker="s", color="firebrick")
ax2.set_xscale("log")
ax2.set_xlabel("Numero de shots (N)")
ax2.set_ylabel("Fracao de repeticoes infactiveis")
ax2.set_title("Viabilidade (Minimize 0): infactibilidade vs. N_shots")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("ruido_shot_noise_sdp.png", dpi=150)
plt.show()



In [ ]:
"""
Extensao do benchmark de tempo (Figura 2) para a formulacao via SDP.

Mesma logica do benchmark da parametrizacao: para cada numero de qubits,
gera um estado aleatorio, monta a base completa de Pauli (necessaria para
tomografia completa), calcula os valores esperados exatos e mede o tempo
que o solver leva para resolver o problema de viabilidade semidefinida.

Ajuste MAX_QUBITS com cuidado: o numero de observaveis (4^n - 1) e o
tamanho das matrizes (2^n x 2^n) crescem rapido, entao comece baixo (ex: 4)
e so aumente depois de ver quanto tempo os primeiros pontos levam.
"""

rng = np.random.default_rng(0)

I2 = np.eye(2, dtype=complex)
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
PAULIS = {"I": I2, "X": sx, "Y": sy, "Z": sz}


def base_pauli_completa(n_qubits):
    """Gera os 4^n - 1 produtos tensoriais de Pauli (exclui a identidade global,
    que ja entra como a restricao de normalizacao Tr(rho) = 1)."""
    observaveis = []
    for combinacao in product("IXYZ", repeat=n_qubits):
        if combinacao == tuple("I" * n_qubits):
            continue
        Qi = PAULIS[combinacao[0]]
        for letra in combinacao[1:]:
            Qi = np.kron(Qi, PAULIS[letra])
        observaveis.append(Qi)
    return observaveis


def estado_aleatorio(d, rng):
    G = rng.normal(size=(d, d)) + 1j * rng.normal(size=(d, d))
    rho = G @ G.conj().T
    return rho / np.trace(rho).real


def resolver_viabilidade_sdp(Q, q, d):
    rho_var = cp.Variable((d, d), hermitian=True)
    constraints = [rho_var >> 0, cp.trace(rho_var) == 1]
    for i in range(len(Q)):
        constraints.append(cp.real(cp.trace(Q[i] @ rho_var)) == q[i])
    problem = cp.Problem(cp.Minimize(0), constraints)
    problem.solve(solver=cp.SCS)
    return problem


MAX_QUBITS = 4  # comece pequeno; cada +1 aumenta MUITO o tempo de execucao

tempos_sdp = []
n_qubits_lista = list(range(1, MAX_QUBITS + 1))

for n in n_qubits_lista:
    d = 2**n
    Q = base_pauli_completa(n)
    rho_alvo = estado_aleatorio(d, rng)
    q = [np.trace(Qi @ rho_alvo).real for Qi in Q]

    inicio = time.time()
    problem = resolver_viabilidade_sdp(Q, q, d)
    duracao = time.time() - inicio

    tempos_sdp.append(duracao)
    print(f"n_qubits={n} | d={d} | M={len(Q)} observaveis | "
          f"status={problem.status} | tempo={duracao:.3f}s")

# --- Grafico -------------------------------------------------------------
plt.figure(figsize=(7, 4.5))
plt.plot(n_qubits_lista, tempos_sdp, marker="o", label="SDP (viabilidade)")

# Se voce ja tem os tempos da parametrizacao (Figura 2) em uma lista,
# por exemplo `tempos_parametrizacao`, descomente e ajuste para sobrepor:
# plt.plot(n_qubits_lista, tempos_parametrizacao[:MAX_QUBITS],
#          marker="s", label="Parametrizacao (T dagger T)")

plt.xlabel("Numero de qubits (n)")
plt.ylabel("Tempo (s)")
plt.title("Tempo de execucao vs. numero de qubits - SDP")
plt.yscale("log")  # escala log ajuda a comparar com crescimento super-exponencial
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("benchmark_tempo_sdp.png", dpi=150)
plt.show()
